# Model Mimarileri ve Karşılaştırmalı Analiz

Bu notebook'ta farklı model mimarilerini karşılaştırmalı olarak inceleyeceğiz. Aynı veri seti üzerinde farklı yapıdaki sinir ağlarının performanslarını değerlendireceğiz.

In [ ]:
# PyTorch ana kütüphanesini içe aktarır - tensor işlemleri ve derin öğrenme için
import torch

# PyTorch'un sinir ağı modülünü içe aktarır - katmanlar, aktivasyon fonksiyonları, kayıp fonksiyonları
import torch.nn as nn

# Stokastik Gradyan İnişi optimizasyonu için
from torch.optim import SGD

# PyTorch veri yükleme yardımcıları - Dataset ve DataLoader sınıfları
from torch.utils.data import Dataset, DataLoader

# Sayısal hesaplamalar ve dizi işlemleri için NumPy kütüphanesi
import numpy as np

# Grafik ve görselleştirme için matplotlib kütüphanesi
import matplotlib.pyplot as plt

# Eğitim/test verisi bölme fonksiyonu
from sklearn.model_selection import train_test_split

# Model performans değerlendirme metrikleri
from sklearn.metrics import accuracy_score

# MNIST veri setini okumak için özel kütüphane
import idx2numpy

### Veri Hazırlığı ve Ön İşleme

MNIST veri setini yükleyip ön işleme adımlarını gerçekleştiriyoruz:
1. Görüntüleri düzleştirme (flatten): 28x28 -> 784 boyutlu vektör
2. Piksel değerlerini normalize etme: [0-255] -> [0-1]
3. Veriyi eğitim ve test setlerine ayırma
4. NumPy dizilerini PyTorch tensor'larına dönüştürme

In [ ]:
# MNIST veri seti dosya yolu
MNIST_DIR = "mnist/"

# MNIST eğitim görüntülerini oku (60000 adet 28x28 piksel görüntü)
# idx2numpy.convert_from_file(): IDX formatındaki dosyayı NumPy dizisine dönüştürür
X_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-images-idx3-ubyte")

# Görüntüleri düzleştir ve normalize et: (60000, 28, 28) -> (60000, 784) ve [0,255] -> [0,1]
# reshape(60000, -1): -1 otomatik boyut hesaplama (28*28=784)
# /255.0: Piksel değerlerini 0-1 aralığına normalize et
X_mnist = X_mnist.reshape(60000, -1) / 255.0

# MNIST eğitim etiketlerini oku (0-9 arası rakam etiketleri)
y_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-labels-idx1-ubyte")

# Veri setini eğitim ve test setlerine böl
# test_size=0.2: %20 test, %80 eğitim
# random_state=42: Tutarlı bölünme için seed
X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42)  

# NumPy dizilerini PyTorch tensörlerine dönüştür
# astype(np.float32): 32-bit float veri tipi (GPU uyumluluğu için)
x = torch.from_numpy(X_train.astype(np.float32))

# Etiketleri int64 tipine dönüştür (CrossEntropyLoss için gerekli)
y = torch.from_numpy(y_train.astype(np.int64))

### Veri Yükleme Altyapısı

PyTorch'un `Dataset` sınıfını kullanarak veri yükleme işlemlerini standartlaştırıyoruz. Bu yaklaşım:
- Bellek yönetimini optimize eder
- Veri yükleme işlemlerini paralel hale getirebilir
- Mini-batch oluşturma işlemlerini otomatikleştirir

In [ ]:
# PyTorch Dataset sınıfından türetilen özel MNIST veri seti sınıfı
class MnistDataset(Dataset):
    def __init__(self, X, y):
        # Veri seti başlangıç fonksiyonu - veri yükleme ve ön işleme
        
        # NumPy dizilerini PyTorch tensörlerine dönüştür
        # astype(np.float32): Özellikler için 32-bit float veri tipi
        self.x = torch.from_numpy(X.astype(np.float32))
        
        # astype(np.int64): Etiketler için 64-bit integer veri tipi (sınıflandırma için)
        self.y = torch.from_numpy(y.astype(np.int64))

    def __getitem__(self, index):
        # Belirli bir indeksteki veri örneğini döndür
        # Bu metod, dataset[index] sözdiziminin çalışmasını sağlar
        # Dönen değer: (özellik_vektörü, etiket) çifti
        return self.x[index], self.y[index]

    def __len__(self):
        # Veri setindeki toplam örnek sayısını döndür
        # Bu metod, len(dataset) sözdiziminin çalışmasını sağlar
        # shape[0]: İlk boyut (örnek sayısı)
        return self.x.shape[0]

### Model Eğitim Fonksiyonu: İleri Seviye Eğitim Dinamikleri

#### Validation-based Training Loop: Generalization Monitoring

Bu fonksiyon, **dual-objective optimization** yaklaşımını benimser: hem eğitim hem de genelleme performansını eş zamanlı olarak izler.

**Kritik Tasarım Kararları:**

#### 1. **Epoch-wise Loss Aggregation Strategy**
```python
# Neden epoch başına ortalama?
batch_train_loss = []  # Her batch'in loss'u
train_loss.append(np.array(batch_train_loss).mean())  # Epoch ortalaması

# Avantajları:
# ✓ Batch size variations'a karşı robust
# ✓ Learning curve smoothing
# ✓ Memory efficient (O(1) storage per epoch)
# ✓ Stable convergence metrics
```

#### 2. **Validation Loss Hesaplama Stratejisi**
```python
with torch.no_grad():  # Gradient computation disabled
    # Memory benefits:
    # - ~50% memory reduction
    # - No computational graph construction
    # - Faster inference (no autograd overhead)
    
    # Consistent evaluation:
    # - Same model state across validation batches
    # - No gradient accumulation side effects
    # - Deterministic results
```

#### 3. **Learning Rate ve Batch Size Analizi**

**lr=0.01 Seçiminin Teorik Temeli:**
```
Optimal Learning Rate Theory (Bottou):
η* ≈ 1 / (μ × L)

Burada:
- μ: Strong convexity parameter 
- L: Lipschitz smoothness constant

MNIST için empirical analysis:
- Feature dimension: 784
- Class count: 10
- Data samples: ~48K

Rule of thumb: η ≈ 1/√d = 1/√784 ≈ 0.036
Conservative choice: η = 0.01 (good starting point)
```

**Batch Size = 256 Optimizasyonu:**
```python
# GPU Memory Analysis:
float32_size = 4 bytes
batch_memory = 256 × 784 × 4 = 802,816 bytes ≈ 0.8MB (input)
gradient_memory = 256 × 10 × 4 = 10,240 bytes ≈ 10KB (output gradients)

# Total memory per batch: ~1MB (very manageable)
# GPU utilization: High (parallel matrix operations)
# Gradient quality: Good (sufficient samples for stable estimates)
```

#### 4. **20 Epoch Convergence Analysis**

**Convergence Rate Theory:**
```
For strongly convex functions (CrossEntropy + SGD):
E[f(θₜ) - f*] ≤ (1 - μη)ᵗ × [f(θ₀) - f*]

With μ ≈ 0.01, η = 0.01:
Convergence rate ≈ (1 - 0.0001)ᵗ ≈ e^(-0.0001t)

For ε = 0.01 accuracy:
t ≥ log(1/ε) / (μη) ≈ log(100) / 0.0001 ≈ 46,000 iterations

With ~190 batches/epoch (48K/256):
Required epochs ≈ 46,000/190 ≈ 242 epochs

20 epochs is conservative but allows comparison of different architectures
```

#### 5. **Model Architecture Comparison Framework**

**Controlled Experimental Design:**
```python
# Constant factors across models:
✓ Same dataset (MNIST)
✓ Same train/test split  
✓ Same batch size (256)
✓ Same optimizer (SGD)
✓ Same learning rate (0.01)
✓ Same number of epochs (20)

# Variable factors:
✗ Model architecture
✗ Activation functions
✗ Number of parameters
```

In [ ]:
# Genel model eğitim fonksiyonu - farklı mimarileri karşılaştırmak için
def train_model(model, train_data_loader, test_data_loader):
    # Çok sınıflı sınıflandırma için kayıp fonksiyonu
    # CrossEntropyLoss = Softmax + NegativeLogLikelihood birleşimi
    criterion = nn.CrossEntropyLoss()
    
    # Stokastik Gradyan İnişi optimizeri
    # model.parameters(): Modelin tüm öğrenilebilir parametrelerini al
    # lr=0.01: Öğrenme hızı (learning rate)
    optimizer = SGD(model.parameters(), lr=0.01)
    
    # Eğitim ve validasyon kayıplarını takip etmek için listeler
    train_loss = []      # Her epoch'taki ortalama eğitim kaybı
    validation_loss = [] # Her epoch'taki ortalama validasyon kaybı

    # 20 epoch boyunca eğitim
    for epoch in range(20):
        # Her epoch için eğitim kayıplarını topla
        batch_train_loss = []
        
        # Eğitim veri setindeki tüm batch'ler üzerinde döngü
        for X, y in train_data_loader:
            # İleri yayılım: Model tahminleri üret
            y_hat = model(X)
            
            # Kayıp hesapla: Tahmin vs gerçek etiketler
            loss = criterion(y_hat, y)
            
            # Geri yayılım: Gradyanları hesapla
            loss.backward()
            
            # Parametreleri güncelle
            optimizer.step()
            
            # Gradyanları sıfırla (bir sonraki iterasyon için)
            optimizer.zero_grad()
            
            # Bu batch'in kaybını kaydet
            batch_train_loss.append(loss.item())
        
        # Bu epoch'un ortalama eğitim kaybını hesapla ve kaydet
        train_loss.append(np.array(batch_train_loss).mean())

        # Validasyon aşaması (gradyan hesaplaması olmadan)
        with torch.no_grad():
            batch_validation_loss = []
            
            # Test veri setindeki tüm batch'ler üzerinde döngü
            for X, y in test_data_loader:
                # Modelden tahmin al
                y_hat = model(X)
                
                # Validasyon kaybını hesapla
                loss = criterion(y_hat, y)
                
                # Bu batch'in validasyon kaybını kaydet
                batch_validation_loss.append(loss.item())

            # Bu epoch'un ortalama validasyon kaybını hesapla ve kaydet
            validation_loss.append(np.array(batch_validation_loss).mean())
        
        # Her 5 epoch'ta ilerleme raporu yazdır
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} done!")
    
    # Eğitim ve validasyon kayıp listelerini döndür
    return train_loss, validation_loss

### Model Mimarisi 1: Tek Katmanlı Lineer Model - Teorik Analiz

#### Linear Classifier'ın Matematiksel Temelleri

**Model Formülasyonu:**
```
f(x) = W^T x + b

Burada:
- W ∈ ℝ^{784×10}: Weight matrix  
- b ∈ ℝ^{10}: Bias vector
- x ∈ ℝ^{784}: Input feature vector (flattened image)
- f(x) ∈ ℝ^{10}: Logits (pre-softmax scores)
```

#### Kapasite Analizi (VC Dimension)

**Vapnik-Chervonenkis Dimension:**
```
VC(Linear Classifier) = d + 1 = 784 + 1 = 785

Interpretation:
- Bu model maksimum 785 örneği "shatter" edebilir
- 785'ten az örnek → perfect memorization possible
- MNIST'te ~48K örnek → Underfitting expected
```

#### Decision Boundary Analizi

**10-Class Linear Separators:**
```python
# Her sınıf için bir hyperplane:
class_i_boundary: w_i^T x + b_i = 0

# Prediction rule:
ŷ = argmax_i (w_i^T x + b_i)

# Voronoi regions in 784D space:
# Space is partitioned into 10 convex regions
```

**Geometric Interpretation:**
```
784-dimensional space → 10 linear decision boundaries
Each boundary: a 783-dimensional hyperplane
Regions: convex polytopes
Limitation: Only linearly separable patterns learnable
```

#### Parameter Initialization Analysis

**Xavier/Glorot Initialization:**
```python
# PyTorch default initialization for nn.Linear:
bound = 1 / √(fan_in) = 1 / √784 ≈ 0.0357

W ~ U(-0.0357, 0.0357)  # Uniform distribution
b ~ U(-0.0357, 0.0357)  # Bias initialization

# Rationale:
# - Maintain activation variance across layers
# - Prevent vanishing/exploding gradients  
# - Balance between too small/too large initial weights
```

#### Loss Landscape Properties

**Convexity Analysis:**
```
For linear model: L(W,b) is convex in (W,b)

Proof sketch:
L(W,b) = (1/n)∑ᵢ CrossEntropy(W^T xᵢ + b, yᵢ)
CrossEntropy is convex in logits
Linear combination preserves convexity
∴ L(W,b) is convex

Implications:
✓ Single global minimum
✓ No local minima
✓ SGD convergence guaranteed
✓ Learning rate less critical
```

#### Representational Limitations

**What Linear Models CAN learn:**
```
✓ Linear combinations of pixel intensities
✓ Template matching (each class = average template)
✓ Principal component projections
✓ Simple geometric features (brightness, contrast)
```

**What Linear Models CANNOT learn:**
```
✗ Non-linear feature combinations
✗ Spatial hierarchies
✗ Translation invariance
✗ Rotation invariance
✗ Complex visual patterns
```

In [ ]:
# Model Mimarisi 1: En Basit Tek Katmanlı Lineer Model
class MNistClassifier_1(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Tek lineer katman: Doğrudan girdiden çıkışa
        # 784: Düzleştirilmiş 28x28 piksel görüntü
        # 10: MNIST'te 10 farklı rakam sınıfı (0-9)
        # Bu model sadece lineer dönüşüm yapar: y = Wx + b
        self.input_layer = nn.Linear(784, 10)
        
    def forward(self, x):
        # İleri yayılım fonksiyonu - sadece lineer dönüşüm
        # Hiçbir aktivasyon fonksiyonu yok (tamamen lineer)
        # Bu model sadece lineer ilişkileri öğrenebilir
        x = self.input_layer(x)
        return x

### Model Mimarisi 2: Sigmoid Aktivasyonlu İki Katmanlı Model - Derinlemesine Analiz

#### Universal Approximation Theorem ve Sigmoid

**Teoretik Temel:**
```
Cybenko (1989) & Hornik (1991) Teoremi:
Tek gizli katmanlı sinir ağı + sigmoid aktivasyon
→ Herhangi bir sürekli fonksiyonu yaklaşabilir

Formal statement:
∀f ∈ C(K), ∀ε > 0, ∃N, wᵢ, bᵢ, αᵢ:
|f(x) - ∑ᵢ₌₁ᴺ αᵢσ(wᵢᵀx + bᵢ)| < ε, ∀x ∈ K

Bu modelde: N = 16 (hidden units)
```

#### Sigmoid Aktivasyon Fonksiyonunun Detaylı Analizi

**Matematiksel Özellikler:**
```python
σ(z) = 1/(1 + e^(-z))

# Critical properties:
Domain: ℝ → (0,1)
Monotonic: dσ/dz > 0 ∀z
Symmetric: σ(-z) = 1 - σ(z)
Differentiable: dσ/dz = σ(z)(1-σ(z))
```

**Saturation Analysis:**
```python
# Saturation zones:
z > 5:  σ(z) ≈ 1,     dσ/dz ≈ 0.007
z < -5: σ(z) ≈ 0,     dσ/dz ≈ 0.007
|z| < 2: Active zone, dσ/dz > 0.1

# Gradient flow implications:
# Large |z| → vanishing gradients
# Network tends to "saturate" during training
```

#### Model Capacity Analysis

**Parameter Count Breakdown:**
```
Layer 1: W₁ ∈ ℝ^{784×16}, b₁ ∈ ℝ^{16}
Layer 2: W₂ ∈ ℝ^{16×10}, b₂ ∈ ℝ^{10}

Total parameters:
784×16 + 16 + 16×10 + 10 = 12,704 parameters

Capacity comparison:
Model 1: 7,840 parameters (linear)
Model 2: 12,704 parameters (62% increase)
```

**Representational Power:**
```python
# Hidden layer representation:
h = σ(W₁ᵀx + b₁) ∈ (0,1)^{16}

# Feature learning capability:
# Each hidden unit learns a "template":
# h_i = σ(w₁ᵢᵀx + b₁ᵢ) represents similarity to template w₁ᵢ

# 16 templates → 16-dimensional feature space
# Final layer: linear combination of templates
```

#### Vanishing Gradient Problem

**Gradient Computation Chain Rule:**
```python
# Backward pass computation:
∂L/∂W₁ = ∂L/∂h × ∂h/∂z₁ × ∂z₁/∂W₁

where:
∂h/∂z₁ = σ'(z₁) = σ(z₁)(1-σ(z₁))

# Saturation effect:
# If σ(z₁) ≈ 0 or σ(z₁) ≈ 1 → σ'(z₁) ≈ 0
# Result: ∂L/∂W₁ ≈ 0 (vanishing gradients)
```

**Saturation Dynamics:**
```python
# Training progression:
# Early epochs: Random weights → σ(z) ∈ (0.2, 0.8) → good gradients
# Mid epochs: Learning → some units saturate
# Late epochs: Many units saturated → slow learning

# Mitigation strategies:
# - Careful weight initialization
# - Lower learning rates
# - Gradient clipping
```

In [ ]:
# Model Mimarisi 2: Sigmoid Aktivasyonlu İki Katmanlı Model
class MNistClassifier_2(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Girdi katmanı: 784 nörondan 16 nörona lineer dönüşüm
        # 16: Gizli katman boyutu (hiperparametre - Model 1'e göre daha küçük)
        self.input_layer = nn.Linear(784, 16)
        
        # Sigmoid aktivasyon fonksiyonu
        # σ(x) = 1/(1+e^(-x)) - çıkışı (0,1) aralığında sıkıştırır
        # Avantajları: Türevi kolay hesaplanır, sürekli ve diferansiyel
        # Dezavantajları: Gradyan kaybı problemi, hesaplama maliyeti
        self.activation = nn.Sigmoid()
        
        # Çıkış katmanı: 16 nörondan 10 sınıfa lineer dönüşüm
        self.output_layer = nn.Linear(16, 10)
        
    def forward(self, x):
        # İleri yayılım fonksiyonu
        
        # Girdi katmanından geçir: 784 -> 16
        x = self.input_layer(x)
        
        # Sigmoid aktivasyonu uygula (non-linearity ekleme)
        # Bu sayede model non-lineer örüntüleri öğrenebilir
        x = self.activation(x)
        
        # Çıkış katmanından geçir: 16 -> 10
        x = self.output_layer(x)
        
        # Logit değerlerini döndür
        return x

### Model Mimarisi 3: ReLU Aktivasyonlu İki Katmanlı Model - Modern Derin Öğrenmenin Temeli

#### ReLU: Deep Learning Devriminin Anahtarı

**Matematiksel Basitlik, Pratik Güç:**
```python
ReLU(z) = max(0, z) = {z  if z > 0
                      {0  if z ≤ 0

# Derivative:
dReLU/dz = {1  if z > 0
           {0  if z ≤ 0

# Comparison with sigmoid:
Sigmoid: 15+ floating point operations
ReLU:    1 comparison + 1 assignment
Speedup: ~15x per activation
```

#### Historical Context ve Impact

**Pre-ReLU Era Problems (2006-2010):**
```
Activation Functions Used:
- Sigmoid: Vanishing gradients
- Tanh: Better than sigmoid, still saturates
- Softmax: Only for output layers

Deep Networks (>3 layers):
- Training was extremely difficult
- Gradients vanished in early layers
- Required layer-wise pre-training
- Limited to shallow architectures
```

**ReLU Revolution (2010-2012):**
```
Krizhevsky et al. (2012) - AlexNet:
✓ Successfully trained 8-layer CNN
✓ ImageNet breakthrough (15.3% → 10.9% error)
✓ ReLU enabled deeper networks
✓ GPU training became practical

Key insights:
- No saturation for positive inputs
- Sparse activation patterns
- Computational efficiency
- Better gradient flow
```

#### Gradient Flow Analysis: ReLU vs Sigmoid

**Comparison of Gradient Propagation:**
```python
# Model 2 (Sigmoid): Chain rule for backprop
∂L/∂W₁ = ∂L/∂h × σ'(z₁) × x
where σ'(z₁) = σ(z₁)(1-σ(z₁)) ∈ [0, 0.25]

# Gradient attenuation: maksimum %25'i geçer

# Model 3 (ReLU): Chain rule for backprop  
∂L/∂W₁ = ∂L/∂h × ReLU'(z₁) × x
where ReLU'(z₁) ∈ {0, 1}

# Gradient flow: %100 geçer (active units için)
```

**Gradient Magnitude Evolution:**
```python
# Sigmoid network (Model 2):
Layer 1 gradient ≈ Layer 2 gradient × 0.25
Result: First layer learns ~4x slower

# ReLU network (Model 3):
Layer 1 gradient ≈ Layer 2 gradient × 1.0  
Result: All layers learn at similar rates
```

#### Sparsity ve Representational Efficiency

**Activation Sparsity:**
```python
# Input distribution: Typically x ~ N(0, σ²)
# After linear transform: z = Wx + b ~ N(μ, Σ)
# ReLU output: h = max(0, z)

# Expected sparsity:
# If z ~ N(0, 1), then P(z > 0) = 0.5
# → ~50% of neurons are active
# → 50% sparse representations
```

**Benefits of Sparsity:**
```
1. Computational Efficiency:
   - Skip computation for inactive neurons
   - Memory bandwidth reduction
   - Cache-friendly access patterns

2. Biological Plausibility:
   - Neurons fire sparsely in brain
   - Energy-efficient processing
   - Robust to noise

3. Representation Quality:
   - Distributed representations
   - Better generalization
   - Less overfitting tendency
```

In [ ]:
# Model Mimarisi 3: ReLU Aktivasyonlu İki Katmanlı Model (Modern Yaklaşım)
class MNistClassifier_3(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Girdi katmanı: 784 nörondan 16 nörona lineer dönüşüm
        # Model 2 ile aynı mimari ama farklı aktivasyon fonksiyonu
        self.input_layer = nn.Linear(784, 16)
        
        # ReLU aktivasyon fonksiyonu
        # ReLU(x) = max(0, x) - negatif değerleri sıfırlar, pozitif değerleri olduğu gibi bırakır
        # Avantajları: Hesaplama açısından çok verimli, gradyan kaybı problemi yok
        # Seyrek aktivasyon sağlar, derin ağlarda daha iyi performans
        self.activation = nn.ReLU()
        
        # Çıkış katmanı: 16 nörondan 10 sınıfa lineer dönüşüm
        self.output_layer = nn.Linear(16, 10)
        
    def forward(self, x):
        # İleri yayılım fonksiyonu
        
        # Girdi katmanından geçir: 784 -> 16
        x = self.input_layer(x)
        
        # ReLU aktivasyonu uygula
        # Bu, modern derin öğrenme uygulamalarında en yaygın kullanılan aktivasyon
        # Gradyan akışını iyileştirir ve eğitimi hızlandırır
        x = self.activation(x)
        
        # Çıkış katmanından geçir: 16 -> 10
        x = self.output_layer(x)
        
        # Logit değerlerini döndür
        return x

### Veri Yükleyicilerin Hazırlanması

Eğitim ve test veri setleri için ayrı DataLoader'lar oluşturuyoruz. Bu yapı:
- Mini-batch eğitimi sağlar
- Veriyi karıştırır (shuffle)
- Paralel veri yüklemeye olanak tanır

In [ ]:
# Eğitim ve test veri yükleyicilerini hazırla

# Eğitim veri seti için Dataset ve DataLoader oluştur
dataset_train = MnistDataset(X_train, y_train)
data_loader_train = DataLoader(
    dataset=dataset_train,   # MnistDataset örneği
    batch_size=256,          # Her batch'te 256 örnek (GPU belleği için optimize)
    shuffle=True             # Her epoch'ta veriyi karıştır (overfitting'i önler)
)

# Test veri seti için Dataset ve DataLoader oluştur
dataset_test = MnistDataset(X_test, y_test)
data_loader_test = DataLoader(
    dataset=dataset_test,    # Test veri seti
    batch_size=256,          # Aynı batch boyutu (tutarlılık için)
    shuffle=True             # Test için karıştırma zorunlu değil ama yapabilir
)

### Kayıp Grafiği Analizi: Model Karşılaştırması için Görsel İntelligence

#### Loss Curve Interpretation Framework

Bu fonksiyon, **training dynamics visualization** için kritik önem taşır. Her çizgi ve eğri, modelin öğrenme sürecinin derinlemesine analizi için veri sağlar.

#### 1. **Dual-Loss Monitoring Strategy**

**Training vs Validation Loss Analysis:**
```python
# Training Loss (Mavi Çizgi):
# - Model'in eğitim verisini ne kadar iyi fit ettiği
# - Optimization algorithm'in effectiveness'i
# - Convergence speed indicator

# Validation Loss (Kırmızı Çizgi):
# - Generalization performance measure
# - Overfitting detection tool
# - Model capacity appropriateness
```

#### 2. **Overfitting Detection Patterns**

**Critical Warning Signs:**
```python
# Healthy Training:
training_loss ↓, validation_loss ↓ (parallel decrease)
Gap: small and stable

# Overfitting Onset:
training_loss ↓, validation_loss ↑ (divergence)
Gap: increasing over time

# Underfitting:
training_loss ↑ (high), validation_loss ↑ (high)
Gap: minimal but both losses high
```

#### 3. **Model Comparison Metrics from Curves**

**Convergence Speed Analysis:**
```python
# Metrics to extract:
def analyze_convergence(train_loss, val_loss):
    # 1. Convergence rate
    initial_loss = train_loss[0]
    final_loss = train_loss[-1]
    improvement_ratio = (initial_loss - final_loss) / initial_loss
    
    # 2. Stability measure
    loss_variance = np.var(train_loss[-5:])  # Last 5 epochs
    
    # 3. Generalization gap
    gap = np.mean(val_loss[-5:]) - np.mean(train_loss[-5:])
    
    # 4. Early convergence detection
    converged_epoch = detect_convergence(train_loss, threshold=0.01)
    
    return {
        'improvement_ratio': improvement_ratio,
        'stability': loss_variance,
        'generalization_gap': gap,
        'convergence_epoch': converged_epoch
    }
```

#### 4. **Expected Curve Patterns per Model**

**Model 1 (Linear) Expected Pattern:**
```
Characteristics:
- Smooth, monotonic decrease
- Fast initial drop (convex optimization)
- Early convergence (~epoch 5-8)
- Small training-validation gap
- Plateau at higher loss value (limited capacity)

Visual signature:
- Steep initial decline
- Quick plateau
- Parallel train/val curves
```

**Model 2 (Sigmoid) Expected Pattern:**
```
Characteristics:
- Slower convergence (vanishing gradients)
- Potential plateaus (saturation)
- Irregular descent (non-convex landscape)
- Moderate training-validation gap
- May show step-wise improvements

Visual signature:
- Gradual decline with flat regions
- Occasional sudden improvements
- More noise in later epochs
```

**Model 3 (ReLU) Expected Pattern:**
```
Characteristics:
- Fastest initial convergence
- Smooth, consistent improvement
- Best final performance
- Potential for slight overfitting
- Most stable optimization

Visual signature:
- Steepest initial slope
- Consistent improvement
- Lowest final loss values
```

In [ ]:
# Kayıp grafiği çizim fonksiyonu - model performanslarını karşılaştırmak için
def plot_loss(train_loss, validation_loss):
    # Grafik boyutunu ayarla
    plt.figure(figsize=(8, 4))
    
    # Eğitim kaybını mavi renkte çiz
    # train_loss: Her epoch'taki ortalama eğitim kaybı listesi
    plt.plot(train_loss, c="b", label="Eğitim")
    
    # Validasyon kaybını kırmızı renkte çiz
    # validation_loss: Her epoch'taki ortalama validasyon kaybı listesi
    plt.plot(validation_loss, c="r", label="Validasyon")
    
    # Lejantı göster (hangi çizginin neyi temsil ettiği)
    plt.legend()
    
    # Grafiği göster
    # Bu grafik sayesinde:
    # - Modelin öğrenme sürecini izleyebiliriz
    # - Overfitting durumunu tespit edebiliriz (validation loss artarken train loss azalıyorsa)
    # - Underfitting durumunu görebiliriz (her iki loss da yüksek kalıyorsa)
    plt.show()

### Model 1: Tek Katmanlı Lineer Model Eğitimi

En basit mimarinin eğitimi ve performans analizi. Beklentimiz:
- Yüksek bias (underfitting)
- Düşük varyans
- Eğitim ve validasyon kayıpları arasında küçük fark

In [ ]:
# Model 1 Eğitimi: Tek Katmanlı Lineer Model
# Bu en basit mimari - sadece lineer dönüşüm, aktivasyon fonksiyonu yok

# Model örneği oluştur
model_1 = MNistClassifier_1()

# Modeli eğit ve eğitim/validasyon kayıplarını al
# train_model fonksiyonu: 20 epoch boyunca eğitim yapar
# Dönen değerler: her epoch'taki ortalama train ve validation loss listeleri
train_loss, validation_loss = train_model(model_1, data_loader_train, data_loader_test)

# Eğitim sürecini görselleştir
# Bu grafik Model 1'in öğrenme sürecini gösterir
# Beklenti: Yüksek bias (underfitting) nedeniyle sınırlı performans
plot_loss(train_loss, validation_loss)

### Model 2: Sigmoid Aktivasyonlu Model Eğitimi

Non-lineer aktivasyon eklemenin etkisini gözlemliyoruz. Beklentimiz:
- Model 1'e göre daha düşük kayıp
- Daha yavaş yakınsama
- Potansiyel gradyan kaybı sorunları

In [ ]:
# Model 2 Eğitimi: Sigmoid Aktivasyonlu İki Katmanlı Model
# Non-linear aktivasyon eklemenin etkisini gözlemleyeceğiz

# Model örneği oluştur
model_2 = MNistClassifier_2()

# Modeli eğit ve sonuçları al
# Bu model gizli katman (16 nöron) ve sigmoid aktivasyon içerir
# Model 1'e göre daha yüksek model kapasitesi bekleniyor
train_loss, validation_loss = train_model(model_2, data_loader_train, data_loader_test)

# Eğitim sürecini görselleştir
# Beklenti: Model 1'e göre daha düşük kayıp değerleri
# Ancak sigmoid'in gradyan kaybı problemi nedeniyle yavaş öğrenme olabilir
plot_loss(train_loss, validation_loss)

### Model 3: ReLU Aktivasyonlu Model Eğitimi

Modern aktivasyon fonksiyonu kullanmanın etkisi. Beklentimiz:
- En hızlı öğrenme
- En düşük kayıp değerleri
- Daha iyi gradyan akışı

In [ ]:
# Model 3 Eğitimi: ReLU Aktivasyonlu İki Katmanlı Model
# Modern derin öğrenme yaklaşımı - ReLU aktivasyon fonksiyonu

# Model örneği oluştur
model_3 = MNistClassifier_3()

# Modeli eğit ve sonuçları al
# Bu model Model 2 ile aynı mimariye sahip ama ReLU aktivasyon kullanır
# ReLU'nun avantajları: hızlı hesaplama, iyi gradyan akışı
train_loss, validation_loss = train_model(model_3, data_loader_train, data_loader_test)

# Eğitim sürecini görselleştir
# Beklenti: Model 2'ye göre daha hızlı yakınsama
# ReLU'nun gradyan akışı iyileştirmesi sayesinde daha iyi performans
plot_loss(train_loss, validation_loss)

In [ ]:
# Karşılaştırmalı Analiz İçin Duplike Import'lar
# (Bu hücre, notebook'un bağımsız çalışabilmesi için gerekli import'ları tekrarlar)

# PyTorch ana kütüphanesi
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import Dataset, DataLoader

# Sayısal hesaplamalar ve görselleştirme
import numpy as np
import matplotlib.pyplot as plt

# Model değerlendirme araçları
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# MNIST veri seti okuma kütüphanesi
import idx2numpy

In [ ]:
# Veri Hazırlığının Tekrarı (Notebook Bağımsızlığı İçin)

# MNIST veri seti yolunu belirle
MNIST_DIR = "mnist/"

# MNIST görüntülerini oku ve ön işle
# 60000 adet 28x28 piksel görüntü -> 60000x784 düzleştirilmiş format
X_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-images-idx3-ubyte")
X_mnist = X_mnist.reshape(60000, -1) / 255.0  # Normalizasyon: [0,255] -> [0,1]

# MNIST etiketlerini oku (0-9 arası rakam etiketleri)
y_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-labels-idx1-ubyte")

# Veri setini eğitim/test olarak böl (%80 eğitim, %20 test)
X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42)  

# PyTorch tensörlerine dönüştür (GPU uyumluluğu için)
x = torch.from_numpy(X_train.astype(np.float32))
y = torch.from_numpy(y_train.astype(np.int64))

In [ ]:
# Dataset Sınıfının Tekrarı (Modülerlik İçin)
class MnistDataset(Dataset):
    def __init__(self, X, y):
        # Veri başlatma: NumPy dizilerini PyTorch tensörlerine dönüştür
        self.x = torch.from_numpy(X.astype(np.float32))  # Özellikler
        self.y = torch.from_numpy(y.astype(np.int64))    # Etiketler

    def __getitem__(self, index):
        # Indeks bazlı veri erişimi: dataset[i] sözdizimini destekler
        return self.x[index], self.y[index]

    def __len__(self):
        # Veri seti uzunluğu: len(dataset) sözdizimini destekler
        return self.x.shape[0]

In [ ]:
# Eğitim Fonksiyonunun Tekrarı (Karşılaştırmalı Analiz İçin)
def train_model(model, train_data_loader, test_data_loader):
    # Eğitim bileşenlerini hazırla
    criterion = nn.CrossEntropyLoss()          # Çok sınıflı sınıflandırma kayıp fonksiyonu
    optimizer = SGD(model.parameters(), lr=0.01)  # SGD optimizeri

    # Kayıp takip listeleri
    train_loss = []      # Eğitim kayıpları
    validation_loss = [] # Validasyon kayıpları

    # 20 epoch eğitim döngüsü
    for epoch in range(20):
        # Eğitim fazı
        batch_train_loss = []
        for X, y in train_data_loader:
            y_hat = model(X)           # İleri yayılım
            loss = criterion(y_hat, y) # Kayıp hesaplama
            loss.backward()            # Geri yayılım
            optimizer.step()           # Parametre güncelleme
            optimizer.zero_grad()      # Gradyan sıfırlama
            batch_train_loss.append(loss.item())
        
        train_loss.append(np.array(batch_train_loss).mean())

        # Validasyon fazı (gradyan hesaplaması olmadan)
        with torch.no_grad():
            batch_validation_loss = []
            for X, y in test_data_loader:
                y_hat = model(X)
                loss = criterion(y_hat, y)
                batch_validation_loss.append(loss.item())
            validation_loss.append(np.array(batch_validation_loss).mean())
        
        # İlerleme raporu (her 5 epoch'ta)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1} done!")
            
    return train_loss, validation_loss

In [ ]:
# Model Sınıflarının Tekrarı - Model 1: Lineer (Basit)
class MNistClassifier_1(nn.Module):
    def __init__(self):
        super().__init__()
        # Tek katmanlı doğrudan bağlantı: 784 -> 10
        # Hiçbir gizli katman ve aktivasyon fonksiyonu yok
        self.input_layer = nn.Linear(784, 10)
        
    def forward(self, x):
        # Sadece lineer dönüşüm: y = Wx + b
        # En basit model - sadece lineer ilişkileri öğrenebilir
        x = self.input_layer(x)
        return x

In [ ]:
# Model 2: Sigmoid Aktivasyonlu (Geleneksel)
class MNistClassifier_2(nn.Module):
    def __init__(self):
        super().__init__()
        # İki katmanlı mimari: 784 -> 16 -> 10
        self.input_layer = nn.Linear(784, 16)  # Gizli katman
        self.activation = nn.Sigmoid()         # Sigmoid aktivasyon
        self.output_layer = nn.Linear(16, 10)  # Çıkış katmanı
        
    def forward(self, x):
        # Katmanlı ileri yayılım:
        x = self.input_layer(x)    # 784 -> 16 lineer dönüşüm
        x = self.activation(x)     # Sigmoid aktivasyon (0,1) aralığı
        x = self.output_layer(x)   # 16 -> 10 lineer dönüşüm
        return x

In [ ]:
# Model 3: ReLU Aktivasyonlu (Modern)
class MNistClassifier_3(nn.Module):
    def __init__(self):
        super().__init__()
        # İki katmanlı mimari: 784 -> 16 -> 10 (Model 2 ile aynı)
        self.input_layer = nn.Linear(784, 16)  # Gizli katman
        self.activation = nn.ReLU()            # ReLU aktivasyon (modern seçim)
        self.output_layer = nn.Linear(16, 10)  # Çıkış katmanı
        
    def forward(self, x):
        # Katmanlı ileri yayılım:
        x = self.input_layer(x)    # 784 -> 16 lineer dönüşüm
        x = self.activation(x)     # ReLU aktivasyon max(0,x)
        x = self.output_layer(x)   # 16 -> 10 lineer dönüşüm
        return x

In [ ]:
# DataLoader'ların Tekrar Hazırlanması

# Eğitim veri yükleyici
dataset_train = MnistDataset(X_train, y_train)
data_loader_train = DataLoader(
    dataset=dataset_train, 
    batch_size=256,  # Mini-batch boyutu
    shuffle=True     # Her epoch'ta karıştır
)

# Test veri yükleyici
dataset_test = MnistDataset(X_test, y_test)
data_loader_test = DataLoader(
    dataset=dataset_test, 
    batch_size=256,  # Aynı batch boyutu
    shuffle=True     # Test için de karıştırma
)

In [ ]:
# Kayıp Görselleştirme Fonksiyonunun Tekrarı
def plot_loss(train_loss, validation_loss):
    # Grafik ayarları
    plt.figure(figsize=(8, 4))
    
    # Eğitim ve validasyon kayıplarını aynı grafikte göster
    plt.plot(train_loss, c="b", label="Eğitim")        # Mavi: eğitim kaybı
    plt.plot(validation_loss, c="r", label="Validasyon")  # Kırmızı: validasyon kaybı
    
    # Lejant ve gösterim
    plt.legend()
    plt.show()
    
    # Bu grafik, modelin performansını değerlendirmek için kritik:
    # - Converge olma durumu
    # - Overfitting tespiti
    # - Model kapasitesi değerlendirmesi

In [ ]:
# KARŞILAŞTIRMALI EĞITIM: Model 1 (Tekrar)
# Lineer model performansını yeniden değerlendirme

# Yeni model örneği oluştur (parametreler rastgele başlatılır)
model_1 = MNistClassifier_1()

# Eğitim sürecini başlat ve kayıpları takip et
# Bu, en basit mimarinin referans performansını belirler
train_loss, validation_loss = train_model(model_1, data_loader_train, data_loader_test)

# Sonuçları görselleştir
# Lineer modelin sınırlarını gözlemleyeceğiz
plot_loss(train_loss, validation_loss)

In [ ]:
# KARŞILAŞTIRMALI EĞITIM: Model 2 (Tekrar)
# Sigmoid aktivasyonunun etkisini tekrar gözlemleme

# Yeni sigmoid model örneği
model_2 = MNistClassifier_2()

# Eğitim ve performans takibi
# Non-linear aktivasyonun Model 1'e göre iyileştirmesini bekleriz
train_loss, validation_loss = train_model(model_2, data_loader_train, data_loader_test)

# Sigmoid modelin öğrenme sürecini görselleştir
# Gradyan kaybı ve yakınsama hızını analiz edelim
plot_loss(train_loss, validation_loss)

In [ ]:
# KARŞILAŞTIRMALI EĞITIM: Model 3 (Tekrar)
# ReLU aktivasyonunun üstünlüğünü doğrulama

# Yeni ReLU model örneği
model_3 = MNistClassifier_3()

# Eğitim ve performans analizi
# ReLU'nun modern derin öğrenmedeki avantajlarını gözlemleyeceğiz
train_loss, validation_loss = train_model(model_3, data_loader_train, data_loader_test)

# ReLU modelin optimizasyon sürecini görselleştir
# En iyi performans ve hızlı yakınsama bekliyoruz
plot_loss(train_loss, validation_loss)

# Sonuç: Üç modelin karşılaştırmalı analizi tamamlandı
# Model 1: Lineer (basit, sınırlı)
# Model 2: Sigmoid (geleneksel, gradyan kaybı riski)
# Model 3: ReLU (modern, optimize edilmiş)